# Taller 3 — Estado hídrico de la planta

**Asignatura:** TAG2027 — Relación Suelo Agua Planta | UDLA
**Unidad:** 3 — El agua en la planta
**RAA:** RAA5
**Duración:** 3 horas
**Ejercicio vinculado:** Ejercicio 2

---

## Objetivos

Al finalizar este taller serás capaz de:
1. Calcular el gradiente de potencial hídrico en el continuo suelo-planta-atmósfera (SPAC)
2. Calcular el déficit de presión de vapor (DPV) y la transpiración a partir de la conductancia estomática
3. Ajustar un modelo de respuesta estomática al potencial hídrico foliar
4. Calcular e interpretar la eficiencia de uso del agua (WUE instantánea e intrínseca)
5. Estimar la extracción de agua por las raíces en un perfil de suelo

---

## Instrucciones

1. Ejecuta cada celda en orden (Shift + Enter)
2. Completa las celdas marcadas con `# TODO`
3. Responde las preguntas de interpretación en celdas de texto
4. Entrega el notebook con todas las celdas ejecutadas

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/frzambra/RSPA-TAG-UDLA/blob/main/labs/taller-03/taller-03.ipynb)

In [ ]:
# ========================================
# Celda de setup — Ejecutar primero
# ========================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12

print('✓ Setup completo.')

---
## Parte 1: Gradiente de potencial hídrico en el SPAC (35 min)

El agua se mueve desde donde el potencial hídrico (Ψ) es **mayor** (menos negativo) hacia donde es **menor** (más negativo). El potencial de la atmósfera depende de la temperatura y la humedad relativa:

$$\Psi_{atm} = \frac{R\,T}{V_w} \cdot \ln\left(\frac{HR}{100}\right)$$

con $R$ = 8.314 J/mol·K, $T$ en Kelvin y $V_w$ = 1.8 × 10⁻⁵ m³/mol. El resultado queda en Pa; dividimos por 10⁶ para obtener MPa.

El flujo de agua se describe con la analogía de la **ley de Ohm**:

$$J = \frac{\Delta\Psi}{R} \quad \Rightarrow \quad R = \frac{\Delta\Psi}{J}$$

### Datos: una vid en dos escenarios

Se midió Ψ en cada compartimento de una vid a mediodía, en un día con T = 25 °C y HR = 50 %, en dos situaciones: **suelo bien regado** y **suelo seco** (10 días sin riego).

In [ ]:
# Potencial hídrico (MPa) en cada compartimento
spac = pd.DataFrame({
    'Compartimento': ['Suelo', 'Raíz', 'Xilema', 'Hoja', 'Atmósfera'],
    'Psi_regado_MPa': [-0.02, -0.30, -0.60, -1.00, np.nan],
    'Psi_seco_MPa':   [-0.80, -1.10, -1.40, -1.80, np.nan]
})

T_aire = 25   # °C
HR_aire = 50  # %

print('Potencial hídrico por compartimento (falta la atmósfera):')
spac

In [ ]:
# TODO 1.1: Calcular el potencial hídrico de la atmósfera
# Crea una función psi_atm(T_C, HR) que devuelva Ψatm en MPa
# y úsala para completar la fila 'Atmósfera' en ambos escenarios

# ----- COMPLETAR -----
# def psi_atm(T_C, HR):
#     R = 8.314        # J/mol·K
#     Vw = 1.8e-5      # m³/mol
#     T_K = ...
#     return ...       # en MPa
#
# psi_a = psi_atm(T_aire, HR_aire)
# spac.loc[spac['Compartimento'] == 'Atmósfera', ['Psi_regado_MPa', 'Psi_seco_MPa']] = psi_a
# print(f'Ψatm = {psi_a:.1f} MPa')
# ---------------------

# spac

In [ ]:
# Celda de validación
if 'psi_atm' in dir():
    ok = np.isclose(psi_atm(25, 50), -95.4, atol=0.5) and np.isclose(psi_atm(14, 75), -38.2, atol=0.5)
    print('✓ Ψatm correcto' if ok else '⚠ Revisa la función psi_atm (¿T en Kelvin? ¿resultado en MPa?)')
else:
    print('⚠ Define la función psi_atm primero')

In [ ]:
# TODO 1.2: Calcular la caída de potencial (ΔΨ) en cada tramo y su % del total
# ΔΨ del tramo = Ψ(compartimento de origen) − Ψ(compartimento de destino)
# Pista: np.diff() calcula destino − origen, así que ΔΨ = -np.diff(...)

# ----- COMPLETAR -----
# tramos = pd.DataFrame({
#     'Tramo': ['Suelo → Raíz', 'Raíz → Xilema', 'Xilema → Hoja', 'Hoja → Atmósfera'],
#     'dPsi_regado': ...,
#     'dPsi_seco': ...
# })
# tramos['Pct_regado'] = tramos['dPsi_regado'] / tramos['dPsi_regado'].sum() * 100
# tramos['Pct_seco'] = ...
# ---------------------

# tramos.round(2)

In [ ]:
# Celda de validación
if 'tramos' in dir():
    ok = (np.allclose(tramos['dPsi_regado'][:3], [0.28, 0.30, 0.40], atol=0.01)
          and np.isclose(tramos['Pct_regado'].iloc[3], 98.97, atol=0.1))
    print('✓ ΔΨ y porcentajes correctos' if ok else '⚠ Revisa el signo y el orden de los tramos')
else:
    print('⚠ Crea el DataFrame tramos primero')

In [ ]:
# Perfil de potencial dentro de la planta (sin la atmósfera, que está fuera de escala)
fig, ax = plt.subplots(figsize=(9, 5))
comp = spac['Compartimento'][:4]
ax.plot(comp, spac['Psi_regado_MPa'][:4], 'o-', color='steelblue', linewidth=2, markersize=8, label='Suelo regado')
ax.plot(comp, spac['Psi_seco_MPa'][:4], 's-', color='darkorange', linewidth=2, markersize=8, label='Suelo seco')
ax.axhline(-1.2, color='red', linestyle='--', alpha=0.6, label='Umbral de estrés en vid (−1.2 MPa)')
ax.set_ylabel('Potencial hídrico Ψ (MPa)')
ax.set_title('Gradiente de potencial suelo → hoja')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# TODO 1.3: Resistencia hidráulica suelo → hoja (ley de Ohm)
# Se midió el flujo de transpiración J en cada escenario:
J_regado = 5.0   # mmol H₂O m⁻² s⁻¹
J_seco = 1.5     # mmol H₂O m⁻² s⁻¹

# R = (Ψsuelo − Ψhoja) / J   [MPa · m² · s / mmol]

# ----- COMPLETAR -----
# R_regado = ...
# R_seco = ...
# ---------------------

# print(f'Resistencia suelo→hoja (regado): {R_regado:.3f} MPa·m²·s/mmol')
# print(f'Resistencia suelo→hoja (seco):   {R_seco:.3f} MPa·m²·s/mmol')
# print(f'La resistencia aumenta {R_seco / R_regado:.1f} veces con el suelo seco')

In [ ]:
# Celda de validación
if 'R_seco' in dir():
    ok = np.isclose(R_regado, 0.196, atol=0.005) and np.isclose(R_seco, 0.667, atol=0.005)
    print('✓ Resistencias correctas' if ok else '⚠ Revisa: R = (Ψsuelo − Ψhoja) / J')
else:
    print('⚠ Calcula R_regado y R_seco primero')

### Pregunta de interpretación

1. ¿En qué tramo se concentra la mayor caída de potencial? ¿Qué estructura la regula?
2. La diferencia Ψsuelo − Ψhoja es casi la misma en ambos escenarios (≈ 1 MPa), pero el flujo con suelo seco es tres veces menor. ¿Qué cambió? ¿Dónde crees que aumentó la resistencia?
3. En el escenario seco, ¿por qué la planta no puede simplemente bajar más su Ψhoja para seguir absorbiendo agua?

---
## Parte 2: Curso diario de la transpiración (40 min)

Se registró cada hora, en una vid de la zona central durante un día de verano, la temperatura (T), la humedad relativa (HR), la conductancia estomática ($g_s$, con porómetro) y el potencial hídrico foliar (Ψhoja, con cámara de Scholander).

### Fórmulas

Presión de vapor de saturación (ecuación de Tetens, FAO-56):

$$e_s = 0.6108 \cdot \exp\left(\frac{17.27\,T}{T + 237.3}\right) \quad [\text{kPa}]$$

Déficit de presión de vapor:

$$DPV = e_s - e_a = e_s \cdot \left(1 - \frac{HR}{100}\right) \quad [\text{kPa}]$$

Transpiración (la forma $E = g_s \times DPV$ vista en clase, con el DPV expresado como fracción de la presión atmosférica $P_{atm}$ ≈ 101.3 kPa):

$$E = g_s \cdot \frac{DPV}{P_{atm}} \times 1000 \quad [\text{mmol H}_2\text{O m}^{-2}\,\text{s}^{-1}]$$

In [ ]:
# Mediciones horarias en vid — día de verano, zona central
dia = pd.DataFrame({
    'Hora':     [7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19],
    'T_C':      [14, 16, 19, 22, 25, 27, 29, 30, 31, 30, 28, 25, 22],
    'HR_pct':   [75, 68, 58, 48, 40, 35, 30, 28, 27, 29, 33, 40, 48],
    'gs_mol':   [0.10, 0.22, 0.30, 0.32, 0.30, 0.25, 0.18, 0.14, 0.12, 0.13, 0.15, 0.12, 0.06],  # mol H₂O m⁻² s⁻¹
    'Psi_hoja': [-0.40, -0.60, -0.80, -1.00, -1.15, -1.30, -1.40, -1.45, -1.45, -1.40, -1.30, -1.10, -0.80]  # MPa
})
P_atm = 101.3  # kPa

dia

In [ ]:
# TODO 2.1: Calcular el DPV de cada hora
# Crea las funciones es(T_C) y dpv(T_C, HR) y agrega la columna 'DPV_kPa'

# ----- COMPLETAR -----
# def es(T_C):
#     return ...
#
# def dpv(T_C, HR):
#     return ...
#
# dia['DPV_kPa'] = dpv(dia['T_C'], dia['HR_pct'])
# ---------------------

# dia[['Hora', 'T_C', 'HR_pct', 'DPV_kPa']].round(2)

In [ ]:
# TODO 2.2: Calcular la transpiración E (mmol H₂O m⁻² s⁻¹) de cada hora
# E = gs × DPV / P_atm × 1000
# Luego identifica la hora de máxima E y la hora de máximo DPV (pista: .idxmax())

# ----- COMPLETAR -----
# dia['E_mmol'] = ...
# hora_E_max = dia.loc[dia['E_mmol'].idxmax(), 'Hora']
# hora_DPV_max = ...
# ---------------------

# print(f'E máxima: {dia["E_mmol"].max():.2f} mmol m⁻² s⁻¹ a las {hora_E_max}:00')
# print(f'DPV máximo: {dia["DPV_kPa"].max():.2f} kPa a las {hora_DPV_max}:00')

In [ ]:
# Celda de validación
if 'E_mmol' in dia.columns and 'DPV_kPa' in dia.columns:
    ok = (np.isclose(dia['DPV_kPa'].max(), 3.28, atol=0.02)
          and np.isclose(dia['E_mmol'].max(), 5.72, atol=0.05)
          and dia.loc[dia['E_mmol'].idxmax(), 'Hora'] == 12)
    print('✓ DPV y E correctos' if ok else '⚠ Revisa las fórmulas de es, DPV y E')
else:
    print('⚠ Calcula DPV_kPa y E_mmol primero')

In [ ]:
# TODO 2.3: Panel de 4 gráficos del curso diario (2 filas × 2 columnas)
# (a) DPV, (b) gs, (c) E, (d) Ψhoja — todos contra la hora
# En el panel de Ψhoja, sombrea los rangos de estrés de la vid con ax.axhspan():
#   > −0.8 sin estrés (verde) | −0.8 a −1.2 estrés leve (amarillo) | < −1.2 estrés (rojo)

# ----- COMPLETAR -----
# fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=True)
#
# axes[0, 0].plot(dia['Hora'], dia['DPV_kPa'], 'o-', color='darkorange', linewidth=2)
# axes[0, 0].set_ylabel('DPV (kPa)')
# axes[0, 0].set_title('(a) Déficit de presión de vapor')
#
# axes[0, 1].plot(...)   # gs
# axes[1, 0].plot(...)   # E
# axes[1, 1].plot(...)   # Ψhoja
# axes[1, 1].axhspan(-0.8, 0, color='green', alpha=0.1, label='Sin estrés')
# ...
# ---------------------

# plt.tight_layout()
# plt.show()

In [ ]:
# TODO 2.4: Clasificar el estado hídrico de cada hora según Ψhoja (umbrales de vid)
# Usa pd.cut() con los límites [-inf, -1.2, -0.8, 0]
# y cuenta cuántas horas pasa la planta en cada estado con .value_counts()

# ----- COMPLETAR -----
# dia['Estado'] = pd.cut(dia['Psi_hoja'],
#                        bins=[-np.inf, -1.2, -0.8, 0],
#                        labels=['Estrés', 'Estrés leve', 'Sin estrés'])
# horas_estado = ...
# ---------------------

# print('Horas en cada estado hídrico:')
# horas_estado

### Pregunta de interpretación

1. ¿A qué hora es máxima la transpiración? ¿Coincide con la hora de máximo DPV? ¿Por qué?
2. Entre las 12:00 y las 16:00 el DPV sigue subiendo, pero E baja. ¿Qué está haciendo la planta y con qué propósito?
3. Si midieras Ψhoja con la cámara de Scholander a las 14:00 de este día, ¿qué recomendarías al productor?

---
## Parte 3: Respuesta estomática al déficit hídrico (35 min)

Durante un ciclo de secado del suelo se midió, en la misma vid, la conductancia estomática a mediodía junto con Ψhoja. La relación sigue una curva **sigmoide**:

$$g_s = \frac{g_{max}}{1 + \exp\left[k\,(\Psi_{50} - \Psi_{hoja})\right]}$$

| Parámetro | Significado |
|-----------|-------------|
| $g_{max}$ | Conductancia máxima (planta sin estrés) |
| $\Psi_{50}$ | Ψhoja al cual gs cae al **50 %** de $g_{max}$ |
| $k$ | Pendiente: qué tan brusco es el cierre estomático |

Vamos a estimar estos parámetros ajustando el modelo a los datos con `curve_fit`.

In [ ]:
# Ciclo de secado: Ψhoja (MPa) y gs (mol H₂O m⁻² s⁻¹) a mediodía
secado = pd.DataFrame({
    'Psi_hoja': [-0.3, -0.5, -0.7, -0.9, -1.0, -1.1, -1.2, -1.3, -1.4, -1.5, -1.7, -1.9],
    'gs_mol':   [0.373, 0.315, 0.343, 0.305, 0.281, 0.246, 0.177, 0.146, 0.091, 0.104, 0.025, 0.010]
})

def modelo_gs(psi, gmax, k, psi50):
    """Respuesta sigmoide de gs a Ψhoja"""
    return gmax / (1 + np.exp(k * (psi50 - psi)))

secado

In [ ]:
# TODO 3.1: Ajustar el modelo sigmoide con curve_fit
# Usa como valores iniciales p0=[0.3, 5, -1.0] (gmax, k, psi50)

# ----- COMPLETAR -----
# params, cov = curve_fit(modelo_gs, ..., ..., p0=[0.3, 5, -1.0])
# gmax, k, psi50 = params
# ---------------------

# print(f'gmax  = {gmax:.3f} mol m⁻² s⁻¹')
# print(f'k     = {k:.2f} MPa⁻¹')
# print(f'Ψ50   = {psi50:.2f} MPa')

In [ ]:
# Celda de validación
if 'psi50' in dir():
    ok = np.isclose(psi50, -1.23, atol=0.05) and np.isclose(gmax, 0.357, atol=0.02)
    print('✓ Ajuste correcto' if ok else '⚠ Revisa los argumentos de curve_fit (x = Psi_hoja, y = gs_mol)')
else:
    print('⚠ Ajusta el modelo primero')

In [ ]:
# TODO 3.2: Graficar datos + curva ajustada
# - Puntos: datos medidos
# - Línea: modelo evaluado en psi_suave = np.linspace(-2.0, -0.2, 100)
# - Línea vertical en Ψ50 y anotación con ax.annotate()
# - Sombrear con ax.axvspan() el rango sin estrés de la vid (Ψhoja > −0.8 MPa)

# ----- COMPLETAR -----
# psi_suave = np.linspace(-2.0, -0.2, 100)
# fig, ax = plt.subplots(figsize=(10, 6))
# ax.scatter(...)
# ax.plot(psi_suave, modelo_gs(psi_suave, gmax, k, psi50), ...)
# ...
# ---------------------

# plt.tight_layout()
# plt.show()

### Pregunta de interpretación

1. ¿Qué significa el valor de Ψ50 que obtuviste? ¿Qué porcentaje de gmax conserva la planta en el umbral de riego de la vid (−1.2 MPa)?
2. ¿Qué implicancia tiene esto para la fotosíntesis y el crecimiento si se espera a que Ψhoja llegue a −1.5 MPa para regar?
3. Un cultivo con Ψ50 = −0.8 MPa y otro con Ψ50 = −1.8 MPa: ¿cuál es más "conservador" con el agua? ¿Cuál tolera mejor la sequía manteniendo la fotosíntesis?

---
## Parte 4: Eficiencia de uso del agua (35 min)

Se midió intercambio gaseoso (fotosíntesis $A$, transpiración $E$ y conductancia $g_s$) en tres cultivos, con riego completo y con déficit hídrico, 3 hojas por tratamiento.

$$WUE = \frac{A}{E} \quad [\mu\text{mol CO}_2 / \text{mmol H}_2\text{O}] \qquad\qquad WUE_i = \frac{A}{g_s} \quad [\mu\text{mol CO}_2 / \text{mol H}_2\text{O}]$$

In [ ]:
# Mediciones de intercambio gaseoso
# A en μmol CO₂ m⁻² s⁻¹ | E en mmol H₂O m⁻² s⁻¹ | gs en mol H₂O m⁻² s⁻¹
gases = pd.DataFrame({
    'Especie': ['Vid'] * 6 + ['Trigo'] * 6 + ['Maíz'] * 6,
    'Tipo': ['C3'] * 12 + ['C4'] * 6,
    'Tratamiento': (['Riego'] * 3 + ['Déficit'] * 3) * 3,
    'A':  [13.5, 14.2, 14.3,  8.8,  9.1,  9.1,
           19.5, 20.3, 20.2, 11.6, 12.2, 12.2,
           34.2, 35.5, 35.3, 24.4, 25.3, 25.3],
    'E':  [3.9, 4.1, 4.0, 2.0, 2.1, 1.9,
           5.4, 5.6, 5.5, 2.7, 2.9, 2.8,
           4.9, 5.1, 5.0, 2.9, 3.1, 3.0],
    'gs': [0.24, 0.26, 0.25, 0.10, 0.11, 0.09,
           0.39, 0.41, 0.40, 0.14, 0.16, 0.15,
           0.21, 0.23, 0.22, 0.10, 0.12, 0.11]
})
gases

In [ ]:
# TODO 4.1: Calcular WUE instantánea (A/E) y WUE intrínseca (A/gs) para cada hoja

# ----- COMPLETAR -----
# gases['WUE'] = ...
# gases['WUEi'] = ...
# ---------------------

# gases.round(2)

In [ ]:
# TODO 4.2: Promediar por especie y tratamiento con groupby
# Agrupa por ['Especie', 'Tipo', 'Tratamiento'] y calcula la media de A, E, gs, WUE y WUEi

# ----- COMPLETAR -----
# resumen_wue = gases.groupby(['Especie', 'Tipo', 'Tratamiento'])[['A', 'E', 'gs', 'WUE', 'WUEi']].mean()
# ---------------------

# print('Promedios por especie y tratamiento:')
# resumen_wue.round(2)

In [ ]:
# Celda de validación
if 'resumen_wue' in dir():
    r = resumen_wue.reset_index().set_index(['Especie', 'Tratamiento'])
    ok = (np.isclose(r.loc[('Vid', 'Riego'), 'WUE'], 3.50, atol=0.05)
          and np.isclose(r.loc[('Maíz', 'Déficit'), 'WUE'], 8.34, atol=0.05)
          and np.isclose(r.loc[('Trigo', 'Riego'), 'WUEi'], 50.0, atol=0.5))
    print('✓ Resumen correcto' if ok else '⚠ Revisa WUE = A/E, WUEi = A/gs y el groupby')
else:
    print('⚠ Crea resumen_wue primero')

In [ ]:
# TODO 4.3: Gráfico de barras agrupadas de WUE por especie y tratamiento
# Pista: resumen_wue['WUE'].unstack('Tratamiento') deja una tabla Especie × Tratamiento
#        que se grafica directamente con .plot(kind='bar', ax=ax)

# ----- COMPLETAR -----
# tabla = resumen_wue['WUE'].droplevel('Tipo').unstack('Tratamiento')[['Riego', 'Déficit']]
# fig, ax = plt.subplots(figsize=(9, 5))
# tabla.plot(kind='bar', ax=ax, color=['steelblue', 'darkorange'], edgecolor='black')
# ...
# ---------------------

# plt.tight_layout()
# plt.show()

### Pregunta de interpretación

1. ¿Qué cultivo es más eficiente en el uso del agua? ¿Se relaciona con su tipo metabólico (C3/C4)?
2. En los tres cultivos, el déficit hídrico **aumenta** la WUE. ¿Significa que conviene regar menos? ¿Qué pasa con la fotosíntesis (A)?
3. ¿Por qué la WUE intrínseca (A/gs) es útil para comparar mediciones hechas en días con distinto DPV?

---
## Parte 5: Absorción de agua por las raíces en el perfil (35 min)

En un huerto de maíz se instalaron sensores FDR a 4 profundidades (cada uno representa una capa de 25 cm). Se regó hasta capacidad de campo el día 0 y se volvió a leer θv el día 4, sin lluvia ni riego entre ambas lecturas y sin drenaje (el suelo ya estaba a CC).

Toda la disminución de θv se debe a la **extracción por raíces** (y evaporación en superficie, que aquí despreciamos):

$$\text{Extracción}_{capa} = (\theta_{v,0} - \theta_{v,4}) \times Z_{capa} \quad [\text{mm}]$$

La **regla 40-30-20-10** dice que, en un perfil dividido en cuartos de la profundidad radical, las raíces extraen ~40 %, 30 %, 20 % y 10 % del agua desde arriba hacia abajo.

In [ ]:
# Lecturas de θv (cm³/cm³) por capa
perfil = pd.DataFrame({
    'Capa': ['0-25 cm', '25-50 cm', '50-75 cm', '75-100 cm'],
    'Espesor_mm': [250, 250, 250, 250],
    'Theta_dia0': [0.320, 0.300, 0.280, 0.260],
    'Theta_dia4': [0.282, 0.271, 0.261, 0.250],
    'Raices_pct': [40, 30, 20, 10]   # regla 40-30-20-10
})
dias = 4
perfil

In [ ]:
# TODO 5.1: Calcular la extracción de agua por capa
# 1. Extraccion_mm = (Theta_dia0 − Theta_dia4) × Espesor_mm
# 2. Extraccion_pct = % de la extracción total que aporta cada capa
# 3. Extracción total (mm) y tasa promedio (mm/día) → estimación de la ETc

# ----- COMPLETAR -----
# perfil['Extraccion_mm'] = ...
# perfil['Extraccion_pct'] = ...
# extraccion_total = ...
# etc_estimada = ...
# ---------------------

# display(perfil[['Capa', 'Extraccion_mm', 'Extraccion_pct', 'Raices_pct']].round(1))
# print(f'Extracción total en {dias} días: {extraccion_total:.1f} mm')
# print(f'ETc estimada: {etc_estimada:.1f} mm/día')

In [ ]:
# Celda de validación
if 'Extraccion_mm' in perfil.columns and 'etc_estimada' in dir():
    ok = (np.allclose(perfil['Extraccion_mm'], [9.5, 7.25, 4.75, 2.5], atol=0.05)
          and np.isclose(etc_estimada, 6.0, atol=0.05))
    print('✓ Extracción correcta' if ok else '⚠ Revisa: extracción = Δθ × espesor (mm)')
else:
    print('⚠ Calcula Extraccion_mm y etc_estimada primero')

In [ ]:
# TODO 5.2: Comparar la extracción medida con la regla 40-30-20-10
# Gráfico de barras horizontales: capas en el eje Y (la superficie arriba),
# % de extracción medida vs % esperado por la regla

# ----- COMPLETAR -----
# fig, ax = plt.subplots(figsize=(9, 5))
# y = np.arange(len(perfil))
# ax.barh(y - 0.2, perfil['Extraccion_pct'], height=0.4, label='Extracción medida', color='steelblue')
# ax.barh(y + 0.2, perfil['Raices_pct'], height=0.4, label='Regla 40-30-20-10', color='lightgray', edgecolor='black')
# ax.set_yticks(y)
# ax.set_yticklabels(perfil['Capa'])
# ax.invert_yaxis()   # superficie arriba
# ...
# ---------------------

# plt.tight_layout()
# plt.show()

### Pregunta de interpretación

1. ¿La extracción medida se ajusta a la regla 40-30-20-10? ¿Qué te dice eso sobre dónde están las raíces activas del maíz?
2. Si tuvieras solo **dos** sensores para programar el riego de este huerto, ¿a qué profundidades los instalarías y por qué?
3. La primera capa pierde agua cuatro veces más rápido que la última. ¿Qué pasa si se riega con láminas pequeñas que solo mojan los primeros 25 cm? ¿Y si se aplican láminas que mojan hasta 1.5 m?

---
## Entregable del Taller 3

Al finalizar, entrega este notebook con todas las celdas completas y ejecutadas.

**Debes incluir:**
1. Ψatm calculado y tabla de ΔΨ por tramo en ambos escenarios
2. Resistencia hidráulica suelo → hoja en ambos escenarios
3. Interpretación: gradiente de potencial y efecto del suelo seco
4. DPV y transpiración horaria, con el panel de 4 gráficos del curso diario
5. Clasificación del estado hídrico por hora e interpretación de la depresión de mediodía
6. Ajuste del modelo gs–Ψhoja (gmax, k, Ψ50) con su gráfico
7. Tabla de WUE e WUEi por cultivo y tratamiento, con su gráfico
8. Interpretación: C3 vs C4 y efecto del déficit hídrico
9. Extracción de agua por capa y ETc estimada, comparada con la regla 40-30-20-10
10. Interpretación: ubicación de sensores y profundidad de riego

**Formato de entrega:** Notebook .ipynb descargado de Colab.

---
*Taller 3 — TAG2027 Relación Suelo Agua Planta | UDLA*